<a href="https://colab.research.google.com/github/AbdarhmanAmin/Car_Evaluation_Project/blob/main/Fake_News_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fake News detection dataset**

Dataset separated in two files:

Fake.csv (23502 fake news article)
True.csv (21417 true news article)

# **Goal of the Project:**

 **The main goal of this project is to build a Machine Learning model capable of automatically detecting whether a news article is Fake or True using textual data.**

# **Read Data From Kaggel**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

In [ ]:
import os

os.listdir(path)

# Text Preprocessing

* Importing essential libraries for data handling, text processing, and machine learning.
* Using **NLTK** to process the Fake News dataset text.
* Tokenization is used to split news into words and sentences.
* Stopwords removal helps eliminate common useless words.
* Stemming and lemmatization are applied to normalize words.
* Converting text into numerical features using word2vec for ML models.

In [ ]:
#required for word2vec
!pip install gensim

Import libraries

In [ ]:
!pip install xgboost
import pandas as pd
import numpy as np
import re
import string
from collections import Counter

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, recall_score, classification_report, accuracy_score, f1_score
from sklearn import metrics

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
true_df = pd.read_csv(f'{path}/True.csv')
fake_df = pd.read_csv(f'{path}/Fake.csv')

In [ ]:
true_df.head(10)

In [ ]:
fake_df.head(5)

**Merge Datasets**

In [ ]:
true_df['target']=1
fake_df['target']=0
df=pd.concat([true_df,fake_df], ignore_index=True)

In [ ]:
df.head(5)

In [ ]:
#merge title and text columns in one column called content
df['content'] = df['title'] + " " + df['text']
df.drop(['title','text'], axis=1, inplace=True)
df.head(5)

removing URLS & Standardize distances






In [ ]:
def basic_clean(text: str) -> str:
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text=re.sub(r'\W', ' ', str(text))
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    return text.strip()

df['content'] = df['content'].apply(basic_clean)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

Lowercasing

In [ ]:
df['content'] = df['content'].str.lower()

Word Tokenization

In [ ]:
df['tokens']=df['content'].apply(word_tokenize)

Stop Words Removal with keeping negative words

In [ ]:
english_stopwords = set(stopwords.words("english")) #set of famous stopwords in english

#removeing negative words from set
for neg_word in ["not", "no", "never","don't","doesn't"]:
    if neg_word in english_stopwords:
        english_stopwords.remove(neg_word)

def remove_stopwords(tokens):
    return [tok for tok in tokens if tok not in english_stopwords]

df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)

Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

def apply_lemmatization(tokens):
    return [lemmatizer.lemmatize(tok) for tok in tokens]


df["lemmatized_tokens"] = df["tokens_no_stop"].apply(apply_lemmatization)

<a id="5"></a>
# <p style="background-color:#E598D8;font-family:newtimeroman;font-size:150%;color:#E1F16B;text-align:center;border-radius:20px 60px;">VECTORIZE</p>

**TF-IDF** in NLP stands for Term Frequency – Inverse document frequency. In NLP cleaned data needs to be converted into a numerical format where each word is represented by a matrix. This is also known as word embedding or Word vectorization.

Term Frequency (TF) = (Frequency of a term in the document)/(Total number of terms in documents)
Inverse Document Frequency(IDF) = log( (total number of documents)/(number of documents with term t))
I will be using TfidfVectorizer() to vectorize the preprocessed data.

**Steps in the Vectorizing:**
* Creating a corpus of lemmatized text
* Converting the corpus in vector form
* Label Encoding the classes in Target

In [ ]:
#Creating a corpus of lemmatized text
#df["corpus"] = df["lemmatized_tokens"].apply(lambda x: " ".join(x))

In [ ]:
#Converting the corpus in vector form
# Use max_features to limit vocabulary size and reduce memory usage
#tfidf = TfidfVectorizer(max_features=20000)  # Limit vocabulary to 5000 most important features
#X = tfidf.fit_transform(df['corpus']).toarray()
#print(f"X shape: {X.shape}")
#print(f"X memory usage: {X.nbytes / 1024**3:.2f} GB")

In [ ]:
#y = df["target"]

# Training Word2Vec Model

In this step, we train a **Word2Vec** model on the cleaned training text.

The model learns word embeddings by:
- Converting words into dense vectors
- Capturing semantic relationships between words
- Using a context window of size 5

These embeddings will be used later as features for the classification model.

In [ ]:
model = Word2Vec(sentences=x_train['lemmatized_tokens'], vector_size=150, window=5, min_count=1, workers=4)

# Sentence Embedding Function

This function converts a list of words (tokens) into a single vector representation using the trained Word2Vec model.

Steps:
- Retrieve word embeddings for each token
- Compute the average of all word vectors in the sentence
- If no valid words exist, return a zero vector

This creates a fixed-size feature representation for each sentence.

In [ ]:
def get_sentence_embedding(tokens, model, vector_size):
    embeddings = []
    for word in tokens:
        if word in model.wv:
            embeddings.append(model.wv[word])
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(vector_size)

# Generating Sentence Embeddings

In this step, we convert all cleaned text into numerical feature vectors using the trained Word2Vec model.

- `train_embedding`: embeddings for training data  
- `test_embedding`: embeddings for testing data  

Each sentence is represented as a 150-dimensional vector by averaging its word embeddings, making the data ready for machine learning models.

In [ ]:
train_embedding = np.array([get_sentence_embedding(tokens, model, 150) for tokens in x_train['cleaned_text']])

In [ ]:
test_embedding = np.array([get_sentence_embedding(tokens, model, 150) for tokens in x_test['cleaned_text']])


# **MODEL BUILDING**

**Steps involved in the Model Building**

* Splitting the testing and training sets
* Build a pipeline of model for four different classifiers.
  1. Naïve Bayes
  2. RandomForestClassifier
  3. KNeighborsClassifier
  4. Support Vector Machines
* Fit all the models on training data
* Get the cross-validation on the training set for all the models for accuracy

In [ ]:
classifiers = [XGBoostClassifier(),
               RandomForestClassifier(),
               SVC()]

for model in classifiers:
    model.fit(test_embedding, y_train)